In [2]:
import pandas as pd

# m5
# ../dataset/m5-forecasting-accuracy/sales_train_evaluation.csv
# online retail
# ../dataset/Online Retail.xlsx

file_path = "../dataset/original_data/202502/2025-02-01.csv"

encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

df = None

for enc in encodings:
    try:
        df = pd.read_csv(file_path, encoding=enc)
        print(f"성공한 인코딩: {enc}")
        break
    except UnicodeDecodeError:
        continue

if df is None:
    raise ValueError("파일 인코딩을 읽지 못했습니다.")

df.info()

성공한 인코딩: cp949
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532141 entries, 0 to 532140
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   collect_day     532141 non-null  object
 1   good_id         532141 non-null  int64 
 2   pum_id          532141 non-null  object
 3   pum_name        532141 non-null  object
 4   good_name       532141 non-null  object
 5   sales_price     532141 non-null  int64 
 6   discount_price  532141 non-null  int64 
 7   benefit         532141 non-null  int64 
dtypes: int64(4), object(4)
memory usage: 32.5+ MB


In [3]:
from pathlib import Path
import pandas as pd

# 원본 데이터 루트 폴더
original_root = Path(r"D:\project\1_final\dataset\original_data")

# 저장할 폴더 / 파일
output_dir = Path(r"D:\project\1_final\dataset")
output_file = output_dir / "integrated_data.csv"

# 남길 품목명
target_pum_names = {
    "라면",
    "즉석식품",
    "소시지",
    "스낵과자",
    "카레",
    "탄산음료",
}

# 읽을 컬럼
use_cols = [
    "collect_day",
    "good_id",
    "pum_id",
    "pum_name",
    "good_name",
    "sales_price",
    "discount_price",
    "benefit",
]

def read_csv_with_fallback(file_path: Path) -> pd.DataFrame:
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(file_path, encoding=enc, usecols=use_cols)
        except Exception as e:
            last_error = e

    raise ValueError(f"파일 읽기 실패: {file_path}\n마지막 에러: {last_error}")

all_dfs = []
csv_files = list(original_root.rglob("*.csv"))

print(f"찾은 CSV 파일 수: {len(csv_files)}")

for file_path in csv_files:
    try:
        df = read_csv_with_fallback(file_path)

        # pum_name 공백 정리
        df["pum_name"] = df["pum_name"].astype(str).str.strip()

        # 원하는 품목만 필터링
        filtered = df[df["pum_name"].isin(target_pum_names)].copy()

        if not filtered.empty:
            all_dfs.append(filtered)

        print(f"[OK] {file_path.name}: {len(filtered)}행 추출")

    except Exception as e:
        print(f"[ERROR] {file_path}: {e}")

if not all_dfs:
    print("조건에 맞는 데이터가 없습니다.")
else:
    integrated_df = pd.concat(all_dfs, ignore_index=True)

    # 중복 제거가 필요하면 사용
    integrated_df = integrated_df.drop_duplicates()

    # 날짜 정렬 시도
    integrated_df["collect_day"] = pd.to_datetime(
        integrated_df["collect_day"], errors="coerce"
    )
    integrated_df = integrated_df.sort_values(
        by=["collect_day", "pum_name", "good_name"],
        ascending=[True, True, True]
    )

    # 가격 컬럼 숫자형 정리
    price_cols = ["sales_price", "discount_price", "benefit"]
    for col in price_cols:
        integrated_df[col] = (
            integrated_df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        integrated_df[col] = pd.to_numeric(integrated_df[col], errors="coerce")

    output_dir.mkdir(parents=True, exist_ok=True)

    # 한글/엑셀 호환 고려해서 utf-8-sig로 저장
    integrated_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n통합 완료")
    print(f"최종 행 수: {len(integrated_df)}")
    print(f"저장 경로: {output_file}")

찾은 CSV 파일 수: 353
[OK] 2025-02-01.csv: 20150행 추출
[OK] 2025-02-02.csv: 20173행 추출
[OK] 2025-02-03.csv: 20127행 추출
[OK] 2025-02-04.csv: 20011행 추출
[OK] 2025-02-05.csv: 20024행 추출
[OK] 2025-02-06.csv: 19969행 추출
[OK] 2025-02-07.csv: 19405행 추출
[OK] 2025-02-08.csv: 19345행 추출
[OK] 2025-02-09.csv: 16641행 추출
[OK] 2025-02-10.csv: 19360행 추출
[OK] 2025-02-11.csv: 19392행 추출
[OK] 2025-02-12.csv: 19412행 추출
[OK] 2025-02-13.csv: 19404행 추출
[OK] 2025-02-14.csv: 19384행 추출
[OK] 2025-02-15.csv: 19358행 추출
[OK] 2025-02-16.csv: 19270행 추출
[OK] 2025-02-17.csv: 19297행 추출
[OK] 2025-02-18.csv: 19335행 추출
[OK] 2025-02-19.csv: 19530행 추출
[OK] 2025-02-20.csv: 19591행 추출
[OK] 2025-02-21.csv: 19612행 추출
[OK] 2025-02-22.csv: 19603행 추출
[OK] 2025-02-23.csv: 17385행 추출
[OK] 2025-02-24.csv: 19557행 추출
[OK] 2025-02-25.csv: 19428행 추출
[OK] 2025-02-26.csv: 19597행 추출
[OK] 2025-02-27.csv: 19226행 추출
[OK] 2025-02-28.csv: 19384행 추출
[OK] 2025-03-01.csv: 19070행 추출
[OK] 2025-03-02.csv: 19110행 추출
[OK] 2025-03-03.csv: 18902행 추출
[OK] 2025-03-04.csv: 1

In [2]:
import pandas as pd

file_path = "../dataset/original_data/202502/2025-02-01.csv"

encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

df = pd.read_csv("../dataset/integrated_data.csv", encoding="utf-8-sig")
df

,collect_day,good_id,pum_id,pum_name,good_name,sales_price,discount_price,benefit
0,2025-02-01,1661351573,A01110,라면,(12개)신라면+너구리 새우탕 김치 육개장사발면 삼양 진 튀김우동 불닭 짜파 오짬 도시락,9800,9800,0
1,2025-02-01,1720659562,A01110,라면,(삼양) 삼양라면멀티 5개입 (1개),10400,10400,0
2,2025-02-01,2104233224,A01110,라면,(올따옴)농심 신라면+오짬+너구리+짜파게티 각5입1봉씩,28900,28900,0
3,2025-02-01,2103424099,A01110,라면,(올따옴)농심 오징어짬뽕5입2봉+신라면5입2봉,27900,27900,0
4,2025-02-01,2104233260,A01110,라면,(올따옴)오뚜기 열라면5입2봉+농심짜파게티5입2봉,25900,25900,0
...,...,...,...,...,...,...,...,...
6013922,2026-02-28,234823273,A02205,탄산음료,환타파인355 쿨피스톡 제로 복숭아 350ml 24입,29700,29700,0
6013923,2026-02-28,234823270,A02205,탄산음료,환타파인355 쿨피스톡 제로 파인애플 350ml 24입,29700,29700,0
6013924,2026-02-28,1912749874,A02205,탄산음료,환타파인500ml 환타 파인 1.5L 6입 + 암바사,35000,35000,0
6013925,2026-02-28,2059558781,A02205,탄산음료,환타포도맛 환타 포도 355ml 24캔,29000,29000,0


In [7]:
product_day_count = (
    df.groupby(["good_id", "pum_name", "good_name"])["collect_day"]
      .nunique()
      .reset_index(name="n_days")
      .sort_values("n_days", ascending=False)
)

print(product_day_count.head(30))
print(product_day_count["n_days"].describe())

           good_id pum_name  \
60089   1141742981     탄산음료   
106943  2121413546       라면   
3391      98479528       라면   
94325   1815201773       라면   
6798     200299789     즉석식품   
6786     199873731       카레   
6691     196052312       카레   
67250   1259101853     스낵과자   
51803    914523570     스낵과자   
2840      81485426     즉석식품   
88326   1626065871       카레   
82776   1496311827     즉석식품   
15245    306763868     즉석식품   
1478      38085964     즉석식품   
107020  2123887944     즉석식품   
83017   1503965930       카레   
63194   1177484930       라면   
82569   1488592945     스낵과자   
51914    918396066       카레   
67335   1262346404     스낵과자   
67334   1262346403     스낵과자   
82453   1483648126     탄산음료   
24409    414157749       카레   
107152  2127285852     탄산음료   
22594    396605884       카레   
83050   1504431308      소시지   
55375   1020613811     즉석식품   
96531   1884583198     탄산음료   
55357   1020103578       라면   
83019   1503967020       카레   

                                      

In [8]:
df["collect_day"] = pd.to_datetime(df["collect_day"], errors="coerce")

summary = (
    df.groupby("good_id")["collect_day"]
      .nunique()
      .reset_index(name="n_days")
)

for threshold in [20, 30, 60]:
    valid_ids = summary.loc[summary["n_days"] >= threshold, "good_id"]
    filtered_df = df[df["good_id"].isin(valid_ids)]

    print(f"\n기준: {threshold}일 이상")
    print(f"남은 상품 수: {filtered_df['good_id'].nunique()}")
    print(f"남은 행 수: {len(filtered_df)}")


기준: 20일 이상
남은 상품 수: 58178
남은 행 수: 5721634

기준: 30일 이상
남은 상품 수: 50076
남은 행 수: 5523668

기준: 60일 이상
남은 상품 수: 27623
남은 행 수: 4577704


In [4]:
_30_out_path = "../dataset/integrated_data_min30days.csv"
_60_out_path = "../dataset/integrated_data_min60days.csv"

df["collect_day"] = pd.to_datetime(df["collect_day"], errors="coerce")

for i in [30, 60]:
    df = df.dropna(subset=["collect_day"]).copy()
    summary = (
    df.groupby("good_id")["collect_day"]
      .nunique()
      .reset_index(name="n_days")
    )
    
    valid_ids = summary.loc[summary["n_days"] >= i, "good_id"]
    
    filtered_df = df[df["good_id"].isin(valid_ids)].copy()
    filtered_df = filtered_df.sort_values(["good_id", "collect_day"]).reset_index(drop=True)

    output_path = _30_out_path if i == 30 else _60_out_path

    filtered_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("30일 이상 등장 상품만 저장 완료")
    print("남은 상품 수:", filtered_df["good_id"].nunique())
    print("남은 행 수:", len(filtered_df))
    print("저장 경로:", output_path)

30일 이상 등장 상품만 저장 완료
남은 상품 수: 50076
남은 행 수: 5523668
저장 경로: ../dataset/integrated_data_min30days.csv
30일 이상 등장 상품만 저장 완료
남은 상품 수: 27623
남은 행 수: 4577704
저장 경로: ../dataset/integrated_data_min60days.csv


In [5]:
input_path = "../dataset/integrated_data_min60days.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")
df["discount_price"] = pd.to_numeric(df["discount_price"], errors="coerce")
df["collect_day"] = pd.to_datetime(df["collect_day"], errors="coerce")

df = df.dropna(subset=["good_id", "discount_price", "collect_day"]).copy()

price_summary = (
    df.groupby("good_id")
      .agg(
          n_days=("collect_day", "nunique"),
          n_unique_prices=("discount_price", "nunique"),
          pum_name=("pum_name", "first"),
          good_name=("good_name", "first"),
      )
      .reset_index()
)

print(price_summary["n_unique_prices"].describe())
print(price_summary["n_unique_prices"].value_counts().sort_index().head(20))

count    27623.000000
mean         4.346016
std          8.066372
min          1.000000
25%          1.000000
50%          1.000000
75%          4.000000
max        146.000000
Name: n_unique_prices, dtype: float64
n_unique_prices
1     16114
2      2821
3      1614
4      1156
5       780
6       675
7       470
8       360
9       314
10      278
11      270
12      233
13      218
14      206
15      187
16      132
17      146
18      140
19      119
20      108
Name: count, dtype: int64


In [7]:
input_path = "../dataset/integrated_data_min60days.csv"
output_path = "../dataset/integrated_data_min60days_price_filtered.csv"
summary_path = "../dataset/integrated_data_min60days_price_filtered_summary.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

df["collect_day"] = pd.to_datetime(df["collect_day"], errors="coerce")
df["discount_price"] = pd.to_numeric(df["discount_price"], errors="coerce")

df = df.dropna(subset=["collect_day", "good_id", "discount_price"]).copy()
df = df.sort_values(["good_id", "collect_day"]).reset_index(drop=True)

summary = (
    df.groupby("good_id")
      .agg(
          n_days=("collect_day", "nunique"),
          n_unique_prices=("discount_price", "nunique"),
          min_price=("discount_price", "min"),
          max_price=("discount_price", "max"),
          pum_name=("pum_name", "first"),
          good_name=("good_name", "first"),
      )
      .reset_index()
)

summary["price_range"] = summary["max_price"] - summary["min_price"]
summary["price_range_rate"] = summary["price_range"] / summary["min_price"].replace(0, pd.NA)

# 추천 기준
filtered_ids = summary.loc[
    (summary["n_unique_prices"] >= 3) &
    (summary["price_range_rate"] >= 0.03),
    "good_id"
]

filtered_df = df[df["good_id"].isin(filtered_ids)].copy()
filtered_df = filtered_df.sort_values(["good_id", "collect_day"]).reset_index(drop=True)

filtered_summary = summary[summary["good_id"].isin(filtered_ids)].copy()
filtered_summary = filtered_summary.sort_values(
    ["n_unique_prices", "price_range_rate", "n_days"],
    ascending=[False, False, False]
)

filtered_df.to_csv(output_path, index=False, encoding="utf-8-sig")
filtered_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("전체 상품 수:", summary["good_id"].nunique())
print("필터 후 상품 수:", filtered_df["good_id"].nunique())
print("필터 후 행 수:", len(filtered_df))
print("저장:", output_path)
print("요약 저장:", summary_path)

전체 상품 수: 27623
필터 후 상품 수: 8291
필터 후 행 수: 1479237
저장: ../dataset/integrated_data_min60days_price_filtered.csv
요약 저장: ../dataset/integrated_data_min60days_price_filtered_summary.csv


In [8]:
input_path = "../dataset/integrated_data_min60days_price_filtered.csv"
summary_path = "../dataset/integrated_data_min60days_price_filtered_summary.csv"

output_path = "../dataset/integrated_data_min60days_price_filtered_clean.csv"
output_summary_path = "../dataset/integrated_data_min60days_price_filtered_clean_summary.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")
summary = pd.read_csv(summary_path, encoding="utf-8-sig")

# 묶음/세트/기획성 상품 제거 패턴
exclude_pattern = r"\+|세트|기획|묶음|번들|골라담기|패키지|모음|혼합|선택|구성"

# summary 기준으로 먼저 상품 제거
summary["good_name"] = summary["good_name"].astype(str).str.strip()
summary_clean = summary[
    ~summary["good_name"].str.contains(exclude_pattern, regex=True, na=False)
].copy()

valid_ids = summary_clean["good_id"].unique()

# 본 데이터도 같은 상품만 남김
df_clean = df[df["good_id"].isin(valid_ids)].copy()

# 정렬
df_clean["collect_day"] = pd.to_datetime(df_clean["collect_day"], errors="coerce")
df_clean = df_clean.sort_values(["pum_name", "good_id", "collect_day"]).reset_index(drop=True)

summary_clean = summary_clean.sort_values(
    ["pum_name", "n_days", "n_unique_prices", "price_range_rate"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

# 저장
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")
summary_clean.to_csv(output_summary_path, index=False, encoding="utf-8-sig")

print("정제 완료")
print("남은 상품 수:", df_clean["good_id"].nunique())
print("남은 행 수:", len(df_clean))
print("저장:", output_path)
print("요약 저장:", output_summary_path)

print("\n카테고리별 남은 상품 수")
print(df_clean.groupby("pum_name")["good_id"].nunique().sort_values(ascending=False))

정제 완료
남은 상품 수: 7589
남은 행 수: 1369026
저장: ../dataset/integrated_data_min60days_price_filtered_clean.csv
요약 저장: ../dataset/integrated_data_min60days_price_filtered_clean_summary.csv

카테고리별 남은 상품 수
pum_name
즉석식품    3912
탄산음료    1537
스낵과자     784
라면       554
카레       509
소시지      331
Name: good_id, dtype: int64


In [12]:
summary_path = "../dataset/integrated_data_min60days_price_filtered_clean_summary.csv"
summary = pd.read_csv(summary_path, encoding="utf-8-sig")

summary["pum_name"] = summary["pum_name"].astype(str).str.strip()
summary["good_name"] = summary["good_name"].astype(str).str.strip()

print("=== 전체 카테고리 분포 ===")
print(summary["pum_name"].value_counts())

print("\n=== unique 카테고리 ===")
print(sorted(summary["pum_name"].unique()))

target_categories = ["라면", "즉석식품", "소시지", "스낵과자", "카레", "탄산음료"]

for category in target_categories:
    temp = summary[summary["pum_name"] == category].copy()
    print(f"\n===== {category} ({len(temp)}개) =====")
    if temp.empty:
        print("해당 카테고리 데이터 없음")
        continue

    temp = temp.sort_values(
        ["n_days", "n_unique_prices", "price_range_rate"],
        ascending=[False, False, False]
    )

    print(
        temp[
            ["good_id", "good_name", "n_days", "n_unique_prices", "min_price", "max_price", "price_range_rate"]
        ].head(10).to_string(index=False)
    )

=== 전체 카테고리 분포 ===
pum_name
즉석식품    3883
탄산음료    1537
스낵과자     784
라면       554
카레       500
소시지      331
Name: count, dtype: int64

=== unique 카테고리 ===
['라면', '소시지', '스낵과자', '즉석식품', '카레', '탄산음료']

===== 라면 (554개) =====
   good_id                             good_name  n_days  n_unique_prices  min_price  max_price  price_range_rate
1455976688                  오뚜기 참깨라면 컵라면 큰컵 110g     353                6        930       1490          0.602151
 457749169                         오뚜기 라면사리 110g     353                5        310        410          0.322581
 304350163       빠르미쇼핑 오뚜기 참깨라면 110g 큰컵 큰사발면 컵라면     353                3       1410       2050          0.453901
1908986810    빠르미쇼핑 농심 새우탕 115g 농심라면 큰컵 큰사발면 컵라면     353                3       1420       1880          0.323944
1909019235 빠르미쇼핑 농심 새우탕 115g 6개 농심라면 큰컵 큰사발면 컵라면     353                3      11200      12200          0.089286
 764506601         짜파게티 짜장라면 농심 올리브 짜파게티 140g 5개     352               33       5920       8320 

In [15]:
from pathlib import Path

summary_path = "../dataset/integrated_data_min60days_price_filtered_clean_summary.csv"
output_dir = Path("../dataset/final_candidates")
output_dir.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(summary_path, encoding="utf-8-sig")

summary["pum_name"] = summary["pum_name"].astype(str).str.strip()
summary["good_name"] = summary["good_name"].astype(str).str.strip()

target_categories = ["라면", "즉석식품", "소시지", "스낵과자", "카레", "탄산음료"]

for category in target_categories:
    temp = summary[summary["pum_name"] == category].copy()

    # 우선순위: 오래 등장 > 가격 다양성 > 가격 변화폭
    temp = temp.sort_values(
        ["n_days", "n_unique_prices", "price_range_rate"],
        ascending=[False, False, False]
    )

    top30 = temp.head(30).copy()

    output_path = output_dir / f"{category}_top30.csv"
    top30.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"{category}: {len(top30)}개 저장 -> {output_path}")

라면: 30개 저장 -> ..\dataset\final_candidates\라면_top30.csv
즉석식품: 30개 저장 -> ..\dataset\final_candidates\즉석식품_top30.csv
소시지: 30개 저장 -> ..\dataset\final_candidates\소시지_top30.csv
스낵과자: 30개 저장 -> ..\dataset\final_candidates\스낵과자_top30.csv
카레: 30개 저장 -> ..\dataset\final_candidates\카레_top30.csv
탄산음료: 30개 저장 -> ..\dataset\final_candidates\탄산음료_top30.csv


In [16]:
from pathlib import Path

summary_path = "../dataset/integrated_data_min60days_price_filtered_clean_summary.csv"
output_dir = Path("../dataset/final_candidates")
output_dir.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(summary_path, encoding="utf-8-sig")

summary["pum_name"] = summary["pum_name"].astype(str).str.strip()
summary["good_name"] = summary["good_name"].astype(str).str.strip()

target_categories = ["라면", "즉석식품", "소시지", "스낵과자", "카레", "탄산음료"]

for category in target_categories:
    temp = summary[summary["pum_name"] == category].copy()

    # 우선순위: 오래 등장 > 가격 다양성 > 가격 변화폭
    temp = temp.sort_values(
        ["n_days", "n_unique_prices", "price_range_rate"],
        ascending=[False, False, False]
    )

    top30 = temp.head(30).copy()

    output_path = output_dir / f"{category}_top30.csv"
    top30.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"{category}: {len(top30)}개 저장 -> {output_path}")

라면: 30개 저장 -> ..\dataset\final_candidates\라면_top30.csv
즉석식품: 30개 저장 -> ..\dataset\final_candidates\즉석식품_top30.csv
소시지: 30개 저장 -> ..\dataset\final_candidates\소시지_top30.csv
스낵과자: 30개 저장 -> ..\dataset\final_candidates\스낵과자_top30.csv
카레: 30개 저장 -> ..\dataset\final_candidates\카레_top30.csv
탄산음료: 30개 저장 -> ..\dataset\final_candidates\탄산음료_top30.csv


In [17]:
import pandas as pd
from pathlib import Path
import html
import re

# ===== 경로 설정 =====
# 네가 직접 고른 카테고리별 파일들이 들어 있는 폴더
selected_dir = Path("../dataset/final_select_goods")

# 원본으로 사용할 데이터
master_path = Path("../dataset/integrated_data_min60days_price_filtered_clean.csv")

# 결과 저장 폴더
output_dir = Path("../dataset/final_selected_data")
output_dir.mkdir(parents=True, exist_ok=True)

# 최종 통합 파일
merged_output_path = output_dir / "final_selected_products_data.csv"
summary_output_path = output_dir / "final_selected_products_summary.csv"


def normalize_text(s):
    s = "" if pd.isna(s) else str(s)
    s = html.unescape(s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s


# ===== 원본 데이터 불러오기 =====
master = pd.read_csv(master_path, encoding="utf-8-sig")

master["pum_name"] = master["pum_name"].astype(str).str.strip()
master["good_name"] = master["good_name"].apply(normalize_text)

if "collect_day" in master.columns:
    master["collect_day"] = pd.to_datetime(master["collect_day"], errors="coerce")

# ===== 선택 파일들 읽기 =====
selected_files = sorted(selected_dir.glob("*.csv"))

if not selected_files:
    raise FileNotFoundError(f"선택 파일이 없습니다: {selected_dir}")

all_selected_ids = set()
summary_rows = []
all_category_frames = []

for file_path in selected_files:
    selected = pd.read_csv(file_path, encoding="utf-8-sig")
    selected.columns = [c.strip() for c in selected.columns]

    if "pum_name" in selected.columns:
        selected["pum_name"] = selected["pum_name"].astype(str).str.strip()
    if "good_name" in selected.columns:
        selected["good_name"] = selected["good_name"].apply(normalize_text)

    category_name = file_path.stem

    # 1순위: good_id로 매칭
    if "good_id" in selected.columns:
        ids = (
            selected["good_id"]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )

        temp = master[master["good_id"].astype(str).isin(ids)].copy()

    # 2순위: good_name + pum_name 조합으로 매칭
    elif {"good_name", "pum_name"}.issubset(selected.columns):
        key_df = selected[["pum_name", "good_name"]].drop_duplicates().copy()

        temp = master.merge(
            key_df,
            on=["pum_name", "good_name"],
            how="inner"
        ).copy()

    else:
        raise ValueError(
            f"{file_path.name} 에 good_id 또는 good_name+pum_name 컬럼이 필요합니다."
        )

    # 카테고리명 보강
    if "pum_name" not in temp.columns or temp["pum_name"].isna().all():
        temp["pum_name"] = category_name

    temp = temp.sort_values(
        ["pum_name", "good_id", "collect_day"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    # 카테고리별 저장
    category_output_path = output_dir / f"{category_name}_selected_data.csv"
    temp.to_csv(category_output_path, index=False, encoding="utf-8-sig")

    # 요약
    n_products = temp["good_id"].nunique() if "good_id" in temp.columns else 0
    n_rows = len(temp)

    summary_rows.append({
        "source_file": file_path.name,
        "category_name": temp["pum_name"].iloc[0] if len(temp) > 0 else category_name,
        "n_products": n_products,
        "n_rows": n_rows,
        "saved_file": category_output_path.name,
    })

    if "good_id" in temp.columns:
        all_selected_ids.update(temp["good_id"].astype(str).unique().tolist())

    all_category_frames.append(temp)

    print(f"[OK] {file_path.name} -> 상품 {n_products}개 / 행 {n_rows}개 저장")


# ===== 전체 통합 =====
merged = pd.concat(all_category_frames, ignore_index=True)
merged = merged.sort_values(
    ["pum_name", "good_id", "collect_day"],
    ascending=[True, True, True]
).reset_index(drop=True)

merged.to_csv(merged_output_path, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

print("\n=== 완료 ===")
print("최종 선택 상품 수:", merged["good_id"].nunique())
print("최종 데이터 행 수:", len(merged))
print("통합 저장:", merged_output_path)
print("요약 저장:", summary_output_path)
print("\n카테고리별 상품 수")
print(merged.groupby("pum_name")["good_id"].nunique())

[OK] 라면_20.csv -> 상품 20개 / 행 6986개 저장
[OK] 소시지_20.csv -> 상품 20개 / 행 7052개 저장
[OK] 스낵과자_20.csv -> 상품 20개 / 행 7049개 저장
[OK] 즉석식품_20.csv -> 상품 20개 / 행 7110개 저장
[OK] 카레_20.csv -> 상품 20개 / 행 7013개 저장
[OK] 탄산음료_20.csv -> 상품 20개 / 행 7047개 저장

=== 완료 ===
최종 선택 상품 수: 120
최종 데이터 행 수: 42257
통합 저장: ..\dataset\final_selected_data\final_selected_products_data.csv
요약 저장: ..\dataset\final_selected_data\final_selected_products_summary.csv

카테고리별 상품 수
pum_name
라면      20
소시지     20
스낵과자    20
즉석식품    20
카레      20
탄산음료    20
Name: good_id, dtype: int64


In [19]:
df = pd.read_csv("../dataset/final_selected_data/라면_20_selected_data.csv", encoding="utf-8-sig")

df

,collect_day,good_id,pum_id,pum_name,good_name,sales_price,discount_price,benefit
0,2025-02-01,17550423,A01110,라면,팔도 틈새라면빨계떡 120G*5입,4840,4840,0
1,2025-02-02,17550423,A01110,라면,팔도 틈새라면빨계떡 120G*5입,4840,4840,0
2,2025-02-03,17550423,A01110,라면,팔도 틈새라면빨계떡 120G*5입,4840,4840,0
3,2025-02-04,17550423,A01110,라면,팔도 틈새라면빨계떡 120G*5입,4840,4840,0
4,2025-02-05,17550423,A01110,라면,팔도 틈새라면빨계떡 120G*5입,4840,4840,0
...,...,...,...,...,...,...,...,...
6981,2026-02-24,2128373290,A01110,라면,오뚜기 참깨라면 용기 110g 8개,11150,11150,0
6982,2026-02-25,2128373290,A01110,라면,오뚜기 참깨라면 용기 110g 8개,11150,11150,0
6983,2026-02-26,2128373290,A01110,라면,오뚜기 참깨라면 용기 110g 8개,11150,11150,0
6984,2026-02-27,2128373290,A01110,라면,오뚜기 참깨라면 용기 110g 8개,11150,11150,0


In [1]:
import pandas as pd

input_path = r"..\dataset\final_selected_data\final_selected_products_data.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

# 날짜형 변환
df["collect_day"] = pd.to_datetime(df["collect_day"], errors="coerce")

# 1) 중복 여부만 확인
dup_mask = df.duplicated(subset=["collect_day", "good_id"], keep=False)
dup_df = df[dup_mask].copy()

print("중복 행 수:", len(dup_df))
print("중복 상품 수:", dup_df["good_id"].nunique())

# 2) 어떤 날짜-상품 조합이 몇 번 중복됐는지 요약
dup_summary = (
    df.groupby(["collect_day", "good_id"])
      .size()
      .reset_index(name="dup_count")
      .query("dup_count > 1")
      .sort_values(["dup_count", "collect_day"], ascending=[False, True])
)

print("\n중복 조합 수:", len(dup_summary))
print(dup_summary.head(20))

# 3) 중복된 원본 행 자세히 보기
dup_detail = df.merge(
    dup_summary[["collect_day", "good_id"]],
    on=["collect_day", "good_id"],
    how="inner"
).sort_values(["collect_day", "good_id"])

print("\n중복 상세 예시")
print(dup_detail.head(20))

중복 행 수: 106
중복 상품 수: 9

중복 조합 수: 53
     collect_day   good_id  dup_count
3     2025-02-01  11508428          2
122   2025-02-02  11508428          2
241   2025-02-03  11508428          2
360   2025-02-04  11508428          2
598   2025-02-06  11508428          2
717   2025-02-07  11508428          2
836   2025-02-08  11508428          2
955   2025-02-09  11508428          2
1069  2025-02-10  11508428          2
1188  2025-02-11  11508428          2
1307  2025-02-12  11508428          2
1546  2025-02-14  11508428          2
1666  2025-02-15  11508428          2
1786  2025-02-16  11508428          2
1903  2025-02-17   9517327          2
1906  2025-02-17  11508428          2
2026  2025-02-18  11508428          2
2145  2025-02-19  11508428          2
2264  2025-02-20  11508428          2
3103  2025-02-27  11508428          2

중복 상세 예시
   collect_day   good_id  pum_id pum_name               good_name  \
18  2025-02-01  11508428  A01918     즉석식품  CJ제일제당 비비고 육개장 500g 1개   
19  2025-02-01  11

In [8]:
import pandas as pd

input_path = "../dataset/final_selected_data/final_selected_products_data.csv"
output_path = "../dataset/duplicated_remove.csv"

df = pd.read_csv(input_path, encoding="utf-8-sig")

# 타입 정리
df["collect_day"] = pd.to_datetime(df["collect_day"], errors="coerce")
df["discount_price"] = pd.to_numeric(df["discount_price"], errors="coerce")

# 기본 결측 제거
df = df.dropna(subset=["collect_day", "good_id", "discount_price"]).copy()

# 문자열 정리
df["pum_name"] = df["pum_name"].astype(str).str.strip()
df["good_name"] = df["good_name"].astype(str).str.strip()

# 상품-날짜 단위로 시장 가격 요약
summary_df = (
    df.groupby(["collect_day", "good_id"])
      .agg(
          pum_id=("pum_id", "first"),
          pum_name=("pum_name", "first"),
          good_name=("good_name", "first"),

          market_min_price=("discount_price", "min"),
          market_mean_price=("discount_price", "mean"),
          market_median_price=("discount_price", "median"),
          market_max_price=("discount_price", "max"),
          market_price_count=("discount_price", "size"),
          market_unique_price_count=("discount_price", "nunique"),
      )
      .reset_index()
)

# 평균가는 보기 좋게 반올림
summary_df["market_mean_price"] = summary_df["market_mean_price"].round(2)

# 정렬
summary_df = summary_df.sort_values(["good_id", "collect_day"]).reset_index(drop=True)

# 저장
summary_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("요약 완료")
print("원본 행 수:", len(df))
print("요약 후 행 수:", len(summary_df))
print("저장 위치:", output_path)
print(summary_df.head())

요약 완료
원본 행 수: 42257
요약 후 행 수: 42204
저장 위치: ../dataset/duplicated_remove.csv
  collect_day  good_id  pum_id pum_name                  good_name  \
0  2025-02-01  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
1  2025-02-02  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
2  2025-02-03  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
3  2025-02-04  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   
4  2025-02-05  9517327  A01918     즉석식품  CJ제일제당 햇반 황금 쌀밥 210g_ 36개   

   market_min_price  market_mean_price  market_median_price  market_max_price  \
0             36900            36900.0              36900.0             36900   
1             36900            36900.0              36900.0             36900   
2             36900            36900.0              36900.0             36900   
3             36900            36900.0              36900.0             36900   
4             36900            36900.0              36900.0             36900   

   market_price_